# Lyrics Dreamer Project

In [42]:
# keep
import os
import random
from functools import partial

import yaml

import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
    AutoModelForCausalLM,
    # AutoModelWithLMHead,
    DataCollatorForLanguageModeling,
    Trainer, 
    TrainingArguments,
    get_cosine_with_hard_restarts_schedule_with_warmup,
)
from datasets import load_dataset, list_datasets, load_from_disk
from pytorch_lightning.loggers import WandbLogger
import wandb

In [2]:
available_ds = [ds for ds in list_datasets() if ds.startswith('huggingartists/')]
available_artists = [ds.split('/')[-1] for ds in available_ds]
print(available_artists)

HfHubHTTPError: 504 Server Error: Gateway Time-out for url: https://huggingface.co/api/datasets

In [6]:
# keep
ARTISTS = [
    'radiohead', 'bob-dylan', 'queen'
]

In [ ]:

os.makedirs("./data_raw", exist_ok=True)
os.makedirs("./data_clean", exist_ok=True)

In [ ]:
ds = load_dataset(f"huggingartists/queen")
ds

In [6]:
ds['train']['text'][0]

'Is this the real life? Is this just fantasy?\nCaught in a landslide, no escape from reality\nOpen your eyes, look up to the skies and see\nIm just a poor boy, I need no sympathy\nBecause Im easy come, easy go, little high, little low\nAny way the wind blows doesnt really matter to me, to me\nMama, just killed a man\nPut a gun against his head, pulled my trigger, now hes dead\nMama, life had just begun\nBut now Ive gone and thrown it all away\nMama, ooh, didnt mean to make you cry\nIf Im not back again this time tomorrow\nCarry on, carry on as if nothing really matters\nToo late, my time has come\nSends shivers down my spine, bodys aching all the time\nGoodbye, everybody, Ive got to go\nGotta leave you all behind and face the truth\nMama, ooh \nI dont wanna die\nI sometimes wish Id never been born at all\nI see a little silhouetto of a man\nScaramouche, Scaramouche, will you do the Fandango?\nThunderbolt and lightning, very, very frightening me\n Galileo, Galileo, Galileo Figaro magnif

For each artist, the data is organised as follows:
- `train`: A Dataset object with only one feature, `text`.
    - `text`: A list of strings, each string being a song.

OK, so we get one row per song. Sometimes, we also get the song listing for each record they made. Maybe we should clean that up. Also, let's remove any row that doesn't contain any text.

In [13]:
# keep
def get_and_clean_data(artist, raw_data_root: str = None, clean_data_root: str = None):
    ds = load_dataset(f"huggingartists/{artist}")
    if raw_data_root is not None:
        os.makedirs("./data_raw", exist_ok=True)
        ds.save_to_disk(os.path.join(raw_data_root, artist))
    # Remove song listings
    ds['train'] = ds['train'].filter(lambda x: not x['text'].startswith('1'))
    # Remote empty sentences
    ds['train'] = ds['train'].filter(lambda x: len(x['text']) > 0)
    if clean_data_root is not None:
        os.makedirs("./data_clean", exist_ok=True)
        ds.save_to_disk(os.path.join(clean_data_root, artist))
    return ds

In [20]:
queen_ds = get_and_clean_data('queen', './data_raw')

Found cached dataset queen (C:/Users/lucfr/.cache/huggingface/datasets/huggingartists___queen/default/1.0.0/709c1ebbe512fe7841002b2c0b3965779d875805bbff8dea3ba68863d957f864)


  0%|          | 0/1 [00:00<?, ?it/s]

Loading cached processed dataset at C:\Users\lucfr\.cache\huggingface\datasets\huggingartists___queen\default\1.0.0\709c1ebbe512fe7841002b2c0b3965779d875805bbff8dea3ba68863d957f864\cache-2d69b2e7637eb1c1.arrow
Loading cached processed dataset at C:\Users\lucfr\.cache\huggingface\datasets\huggingartists___queen\default\1.0.0\709c1ebbe512fe7841002b2c0b3965779d875805bbff8dea3ba68863d957f864\cache-26d8a8cc5d6c3ae5.arrow


Saving the dataset (0/1 shards):   0%|          | 0/551 [00:00<?, ? examples/s]

In [19]:
# keep
def split_datasets(ds = None, valid_size = 0.1, load_dir: str = None, save_data_root: str = None):
    if ds is None:
        if load_dir is None:
            raise ValueError("Either ds or load_dir must be provided.")
        ds = load_from_disk(load_dir)
    ds_split = ds['train'].train_test_split(test_size=valid_size)
    ds_split['validation'] = ds_split.pop('test')
    if save_data_root is not None:
        os.makedirs(save_data_root, exist_ok=True)
        ds_split.save_to_disk(os.path.join(save_data_root, artist))
    return ds_split

In [27]:
split_ds = split_datasets(load_dir='./data_raw/queen', valid_size=0.1)
type(split_ds['train'][0])

dict

In [16]:
# keep

# Now we load the tokenizer and pretrained model to perform text generation. We chose `distilgpt2` because it's smaller and faster to train.
checkpoint = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
# Add the EOS token as PAD token to avoid warnings and allow PyTorch to create tensors
tokenizer.pad_token = tokenizer.eos_token

Simply padding and truncating deletes any tokens that are longer than the maximum length:

In [12]:
test_tokens = tokenizer(ds['train']['text'], return_tensors='pt', padding=True, truncation=True, max_length=256)
test_tokens['input_ids'].shape

torch.Size([580, 256])

An alternative is to use the `return_overflowing_tokens` argument, which returns the tokens that were truncated. We can then use the `stride` argument to get overlapping sequences.

In [13]:
test_tokens = tokenizer(ds['train']['text'], return_tensors='pt', padding=True, truncation=True, max_length=256, stride=128, return_overflowing_tokens=True)
test_tokens

{'input_ids': tensor([[ 3792,   428,   262,  ...,   845,    11,   845],
        [  198,  1532,  1846,  ...,   198, 29926,  2611],
        [23101,   502,   198,  ..., 50256, 50256, 50256],
        ...,
        [32274,   286,   616,  ...,   198, 31442,    11],
        [ 1613,   262, 10329,  ..., 50256, 50256, 50256],
        [   33, 21584,    11,  ..., 50256, 50256, 50256]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'overflow_to_sample_mapping': tensor([  0,   0,   0,  ..., 578, 578, 579])}

In [43]:
# Let's double-check the overlaps
for i in range(len(test_tokens['input_ids'])):
    input_ids = test_tokens['input_ids'][i]
    attention_mask = test_tokens['attention_mask'][i]
    print(f"Segment {i}:")
    print(f"Last {128} tokens: {tokenizer.decode(input_ids[-128:], skip_special_tokens=True)}")
    if i < len(test_tokens['input_ids']) - 1:
        next_input_ids = test_tokens['input_ids'][i + 1]
        next_attention_mask = test_tokens['attention_mask'][i + 1]
        print(
            f"First {128} tokens of next segment: {tokenizer.decode(next_input_ids[:128], skip_special_tokens=True)}")


Segment 0:
Last 128 tokens: , let it be, let it be, let it be
Yeah, there will be an answer, let it be
Let it be, let it be, let it be, let it be
Whisper words of wisdom, let it be
Let it be, let it be, let it be, yeah, let it be
Whisper words of wisdom, let it be
And when the night is cloudy, there is still a light that shines on me
Shine on til tomorrow, let it be
I wake up to the sound of music, Mother Mary comes to me
Speaking words of wisdom,
First 128 tokens of next segment: , let it be, let it be, let it be
Yeah, there will be an answer, let it be
Let it be, let it be, let it be, let it be
Whisper words of wisdom, let it be
Let it be, let it be, let it be, yeah, let it be
Whisper words of wisdom, let it be
And when the night is cloudy, there is still a light that shines on me
Shine on til tomorrow, let it be
I wake up to the sound of music, Mother Mary comes to me
Speaking words of wisdom,
Segment 1:
Last 128 tokens:  Let it be
Let it be, let it be, let it be, yeah, let it be
Oh

Seems correct.

In [14]:
test_tokens['input_ids'].shape

torch.Size([1090, 256])

However, the most memory efficient way to do this is to let the DataCollator handle the padding, because this ensure minimal-length of each batch. So we deactivate padding in the function below. It won't be able to return tensors anymore, because of the varying length of the sequences, but the DataCollector will take care of that.

In [ ]:
# keep
def tokenize(examples: dict, block_size: int = 128, stride: int = 128):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=block_size,
        stride=stride,
        padding=False,
        return_overflowing_tokens=True,
    )
def load_preprocess_and_save(
        tokenizer_function: callable,
        block_size: int = 256,
        stride: int = 128,
        valid_size: float = 0.1,
        save_dir: str = './data_tokenized',
        save_raw: bool = False,
        save_clean: bool = False,
        save_split: bool = False,
        artist_list: list = ARTISTS,
):
    tokenized_datasets = {}
    os.makedirs(save_dir, exist_ok=True)

    for artist in artist_list:
        print(f"Processing {artist}")
        ds = get_and_clean_data(
            artist,
            raw_data_root='./data_raw' if save_raw else None,
            clean_data_root='./data_clean' if save_clean else None,

        )
        ds = split_datasets(
            ds,
            valid_size=valid_size,
            save_data_root='./data_split' if save_split else None,
        )
        tokenizer_partial = partial(tokenizer_function, block_size=block_size, stride=stride)
        tokenized_datasets[artist] = ds.map(tokenizer_partial, batched=True, remove_columns=['text'])
        tokenized_datasets[artist].remove_columns(['overflow_to_sample_mapping'])
        tokenized_datasets[artist].save_to_disk(os.path.join(save_dir, artist))
    return tokenized_datasets

#
# def group_inputs(examples: dict):
#     concat_inputs = {k: [x for x in examples[k]] + [] for k in examples.keys()}
#     total_length = len(concat_inputs[list(examples.keys())[0]])
#     total_length = (total_length // block_size) * block_size
#     result = {
#         k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
#         for k, t in concat_inputs.items()
#     }
#     result["labels"] = result["input_ids"].copy()
#     return result


In [36]:
%%writefile ./config/train_config.yaml

pretrained_model: 'gpt2'
block_size: 256
stride: 128
batch_size: 8
n_training_steps: 3000
valid_size: 0.1

Overwriting ./config/train_config.yaml


In [39]:
# keep
# Read configuration from YAML file
with open('./config/train_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

tokenized_datasets = load_preprocess_and_save(
    tokenizer_function=tokenize,
    block_size=config['block_size'],
    stride=config['stride'],
    valid_size=0.1,
    save_dir='./data_tokenized',
)

Processing radiohead


Found cached dataset radiohead (C:/Users/lucfr/.cache/huggingface/datasets/huggingartists___radiohead/default/1.0.0/d37c36412fe443a2baba285a9d3ab4e8e4df79db8feb93b2f775c3dacfeca9eb)


  0%|          | 0/1 [00:00<?, ?it/s]

Loading cached processed dataset at C:\Users\lucfr\.cache\huggingface\datasets\huggingartists___radiohead\default\1.0.0\d37c36412fe443a2baba285a9d3ab4e8e4df79db8feb93b2f775c3dacfeca9eb\cache-1deaac70c10cad4c.arrow
Loading cached processed dataset at C:\Users\lucfr\.cache\huggingface\datasets\huggingartists___radiohead\default\1.0.0\d37c36412fe443a2baba285a9d3ab4e8e4df79db8feb93b2f775c3dacfeca9eb\cache-52bafcb61b995f7e.arrow


Map:   0%|          | 0/373 [00:00<?, ? examples/s]

...

Loading cached processed dataset at C:\Users\lucfr\.cache\huggingface\datasets\huggingartists___queen\default\1.0.0\709c1ebbe512fe7841002b2c0b3965779d875805bbff8dea3ba68863d957f864\cache-2d69b2e7637eb1c1.arrow
Loading cached processed dataset at C:\Users\lucfr\.cache\huggingface\datasets\huggingartists___queen\default\1.0.0\709c1ebbe512fe7841002b2c0b3965779d875805bbff8dea3ba68863d957f864\cache-26d8a8cc5d6c3ae5.arrow


Map:   0%|          | 0/495 [00:00<?, ? examples/s]

Map:   0%|          | 0/56 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/954 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/106 [00:00<?, ? examples/s]

In [43]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, return_tensors='pt')

In [47]:
# import sys
os.environ['WANDB_DISABLED'] = "true"

# wandb.init(project="lyrics-dreamer", name='sequential-training')
# logger = WandbLogger(save_dir='./logs')

for artist, dataset in tokenized_datasets.items():
    print(f"Training on {artist}:")
    model = AutoModelForCausalLM.from_pretrained(checkpoint)
    training_args = TrainingArguments(
        output_dir=f'./checkpoints/{artist}',
        learning_rate=5e-5,
        lr_scheduler_type='cosine_with_restarts',
        save_strategy='epoch',
        save_total_limit=1,
        per_device_train_batch_size=config['batch_size'],
        push_to_hub=False,
        no_cuda=False,
        fp16=False,
        logging_steps=50,
        num_train_epochs=3,
        weight_decay=1e-2,
        # report_to='wandb',
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=training_args.learning_rate)
    lr_scheduler = get_cosine_with_hard_restarts_schedule_with_warmup(
        optimizer,
        num_warmup_steps=10,
        num_training_steps=training_args.num_train_epochs * len(dataset['train']) // training_args.per_device_train_batch_size,
        num_cycles=1,
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset['train'],
        data_collator=data_collator,
        optimizers=(optimizer, lr_scheduler),
    )
    trainer.train()

Training on radiohead:


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Step,Training Loss


Training on bob-dylan:


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Step,Training Loss


Training on queen:


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Step,Training Loss


## Now let's generate some lyrics!


In [293]:
artist = 'radiohead'
checkpoint_dir = f"./checkpoints/{artist}"
checkpoint_name = sorted(os.listdir(checkpoint_dir))[0]
checkpoint_name

'checkpoint-3024'

In [294]:
# Load the model
model = AutoModelForCausalLM.from_pretrained(os.path.join(checkpoint_dir, checkpoint_name))
model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [295]:
prompt = "Once upon a time, you dressed so fine"
seed = 123
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [296]:
encoded_prompt = tokenizer(prompt, return_tensors='pt')

In [297]:
output = model.generate(
    input_ids=encoded_prompt['input_ids'],
    attention_mask= encoded_prompt['attention_mask'],
    min_length=100,
    max_length=250,
    temperature=1.,
    repetition_penalty=1.1,
    top_p=0.98,
    # top_k=50,
    do_sample=True,
    num_return_sequences=5,
)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [299]:
generated_lyrics = [tokenizer.decode(out, skip_special_tokens=True) for out in output]

for lyric in generated_lyrics:
    print(lyric)
    print("-------")


Once upon a time, you dressed so fine
As the saying goes
And I stood in front of your face
While listening to your heart
Open wide as you smile
Im such a tease, and youre such a flirt
Once upon a time, you washed up from your dives
Splinters all over the beach
And now Im back where I started
With my reflection in the glass, reflection in the glass
Its always best when the light is off
Its always better on the outside
Fifteen blows to the brain
Eight blows to the backofthe head
Obligations
Complications
Routines and schedules
A job that killed you, killed you
Bird that’s flown into my room
Tied to a branch
Tied round my neck
Before I even start shooting
Finish what I am
Finish what I must
I dont understand
Blame it on the black star
My eyes are on the wall
Too scared to talk
Etc. blame it on the falling sky
Ah, blame it on Amazon
The raindrops and the flies
The fish eat the plants
The birds eat me alive
I dont know why I bleed
Youve no
-------
Once upon a time, you dressed so fine
And a